# DA6401 Assignment 2 — Final Training

Run top to bottom. **T4 GPU required.**

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — change runtime')

In [ ]:
# clone and setup
import os
if not os.path.exists('/content/da6401_assignment_2'):
    !git clone https://github.com/Mohmad-Yaqoob/da6401_assignment_2.git
os.chdir('/content/da6401_assignment_2')
!git pull origin main
!pip install -q wandb albumentations scikit-learn
print(os.listdir('.'))

In [ ]:
# mount drive and restore checkpoints if any
from google.colab import drive
import shutil, os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/da6401_a2_final'
LOCAL = '/content/da6401_assignment_2/checkpoints'
os.makedirs(DRIVE, exist_ok=True)
restored = []
for f in os.listdir(DRIVE):
    if f.endswith('.pth'):
        shutil.copy(os.path.join(DRIVE,f), os.path.join(LOCAL,f))
        restored.append(f)
print('restored:', restored or 'nothing yet')
print('local checkpoints:', os.listdir(LOCAL))

In [ ]:
def save_to_drive():
    DRIVE = '/content/drive/MyDrive/da6401_a2_final'
    LOCAL = '/content/da6401_assignment_2/checkpoints'
    import shutil, os
    for f in os.listdir(LOCAL):
        if f.endswith('.pth'):
            shutil.copy(os.path.join(LOCAL,f), os.path.join(DRIVE,f))
            print('saved:', f)
print('save_to_drive() ready')

In [ ]:
import wandb
wandb.login()

In [ ]:
# sanity check dataset
import os, sys
os.chdir('/content/da6401_assignment_2')
sys.path.insert(0, '/content/da6401_assignment_2')
from data.pets_dataset import prepare_dataset, get_dataloaders
root = prepare_dataset('./pets_data')
tr, va, te = get_dataloaders(root=root, batch_size=8)
b = next(iter(tr))
print('image:', b['image'].shape, b['image'].dtype)
print('label:', b['label'])
print('bbox:', b['bbox'][0])
print('mask:', b['mask'].shape, b['mask'].unique())
print('OK')

## Task 1 — Classification

Run 3 times for dropout ablation (section 2.2 of report).

In [ ]:
os.chdir('/content/da6401_assignment_2')
!python train.py --task cls --epochs 40 --lr 1e-4 --batch_size 32 --dropout_p 0.0

In [ ]:
save_to_drive()

In [ ]:
os.chdir('/content/da6401_assignment_2')
!python train.py --task cls --epochs 40 --lr 1e-4 --batch_size 32 --dropout_p 0.2

In [ ]:
save_to_drive()

In [ ]:
# this is the keeper — saves classifier.pth used by tasks 2 and 3
os.chdir('/content/da6401_assignment_2')
!python train.py --task cls --epochs 40 --lr 1e-4 --batch_size 32 --dropout_p 0.3

In [ ]:
save_to_drive()

## Task 2 — Localization

In [ ]:
os.chdir('/content/da6401_assignment_2')
!python train.py --task loc --epochs 30 --lr 5e-4 --batch_size 32

In [ ]:
save_to_drive()

## Task 3 — Segmentation (3 strategies for W&B section 2.3)

In [ ]:
os.chdir('/content/da6401_assignment_2')
!python train.py --task seg --epochs 30 --lr 1e-3 --batch_size 16 --strategy frozen

In [ ]:
save_to_drive()

In [ ]:
os.chdir('/content/da6401_assignment_2')
!python train.py --task seg --epochs 30 --lr 1e-3 --batch_size 16 --strategy partial

In [ ]:
save_to_drive()

In [ ]:
# full fine-tuning — this is the one that saves unet.pth
os.chdir('/content/da6401_assignment_2')
!python train.py --task seg --epochs 30 --lr 1e-3 --batch_size 16 --strategy full

In [ ]:
save_to_drive()

## Task 4 — Multi-task joint fine-tuning

In [ ]:
os.chdir('/content/da6401_assignment_2')
!python train.py --task multi --epochs 20 --lr 5e-4 --batch_size 16

In [ ]:
save_to_drive()

## Verify checkpoints are correct format

In [ ]:
import torch, os
os.chdir('/content/da6401_assignment_2')
for name in ['classifier.pth', 'localizer.pth', 'unet.pth']:
    path = f'checkpoints/{name}'
    if os.path.exists(path):
        ckpt = torch.load(path, map_location='cpu')
        keys = list(ckpt.keys()) if isinstance(ckpt, dict) else 'plain state_dict'
        state = ckpt.get('state_dict', ckpt)
        sample_keys = list(state.keys())[:3]
        print(f'{name}: top-level keys={keys}, state sample={sample_keys}')
    else:
        print(f'{name}: NOT FOUND')

## Test MultiTaskPerceptionModel loads correctly

In [ ]:
import torch, os, sys
os.chdir('/content/da6401_assignment_2')
sys.path.insert(0, '/content/da6401_assignment_2')

# simulate what autograder does — copy checkpoints to root
import shutil
for f in ['classifier.pth', 'localizer.pth', 'unet.pth']:
    shutil.copy(f'checkpoints/{f}', f)

# temporarily patch gdown to skip download (files already exist)
import unittest.mock as mock
with mock.patch('gdown.download', return_value=None):
    from multitask import MultiTaskPerceptionModel
    model = MultiTaskPerceptionModel()

model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
x = torch.randn(4, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(x)
print('classification:', out['classification'].shape)
print('localization:  ', out['localization'].shape)
print('segmentation:  ', out['segmentation'].shape)

# check F1 on val set
from data.pets_dataset import get_dataloaders
from sklearn.metrics import f1_score
_, va, _ = get_dataloaders(root='./pets_data', batch_size=32)
preds, lbls = [], []
with torch.no_grad():
    for b in va:
        o = model(b['image'].to(device))
        preds.extend(o['classification'].argmax(1).cpu().numpy())
        lbls.extend(b['label'].numpy())
print('val F1:', f1_score(lbls, preds, average='macro', zero_division=0))